# An Introduction to NTQR -- The Python Package for the Logic of Unsupervised Evaluation of Classifiers

NTQR is the Python package for the logic of unsupervised evaluation of classifiers. This notebook gives you a quick introduction to its use for your own work.

## What is unsupervised evaluation of classifiers?

There can be many definitions of unsupervised evaluation. NTQR defines it quite simply. Given the counts of the joint decisions of $N$ classifiers labeling $Q$ items, each with one of $R$ labels, what joint evaluations are logically consistent with it?

This is a minimalist interpretation of the unsupervised task. It assumes you have no knowledge of the test domain or even what the labels mean. All you have are their joint decisions on the labels for the $Q$ items.

## Logical consistency between disagreeing experts is our only tool

The algorithms in NTQR define a logic of unsupervised evaluation because its only tool is logical consistency between disagreeing experts. For example, if you and I label some items and we disagree in our decisions, we cannot both be 100% correct. NTQR generalizes this idea. For any test of size $Q$ we can compute two sets, the possible and consistent ones, for an arbitrary set of $N$ classifiers labeling $Q$ items with $R$ labels.

## The possible set of evaluations

The possible set of evaluations for a test of size $Q$ that $N$ classifiers have taken is enormous even for small tests. NTQR has the `ntqr.evaluations.PossibleSet` class that allows you to generate the sets exactly or by random sampling. You can define your own symbols for the labels and classifiers that NTQR will use when it constructs its axiomatic equations. These tuples also define, implicitly, the order of the variables used for the unknown label response counts of the classifiers.

In [1]:
import ntqr.evaluations

# We will take the case of three labels, R=3
labels = ('a', 'b', 'c')
# And N=4 classifiers
classifiers = ('i', 'j', 'k', 'l')
possible_set = ntqr.evaluations.PossibleSet(labels, classifiers)
possible_set

PossibleSet(('a', 'b', 'c'),('i', 'j', 'k', 'l'))

The possible set is quite large and you rarely need it. But, there is a function that can compute its size exactly. To use it, you must specify the Q-point you are assuming for the unknown answer key. NTQR algorithms are parametrized by the Q-point. This is just the prevalence of the true labels in the unknown answer key.

In [2]:
# Suppose these classifiers are taking a Q=30 test
# Let's pick a random point in the Q-simplex for this test
import numpy as np

# We want label counts in the answer key that sum to 30
# This is one of NTQR's axioms!
Q = 30

# Let's pick uniformly so we land somewhere near equal
# label counts in the answer key
probabilities = [1/3, 1/3, 1/3]

# Draw a single sample of 3 integers
q_point = np.random.multinomial(Q, probabilities)
print(q_point)
set_count = possible_set.set_count_at_ql(q_point)
print(set_count, float(set_count))

[10 10 10]
187212612555214866270168836241735501327 1.8721261255521488e+38


We can see that the possible set is huge!

Let's randomly sample the possible set so you can see what NTQR joint evaluations look like. NTQR contains both exact and random generators of the possible set.

In [3]:
import itertools
possible_evals = possible_set.random_points(q_point, 2)
# We should get two joint evaluations, each with three
# tuples, one for each label, and each label tuple 3**4=81 long.
for eval in possible_evals:
    print(eval)
    print()

((1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, np.int64(0)), (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, np.int64(0)), (1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, np.int64(0)))

((0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 2, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 

In [4]:
# These evaluations are members of the possible set for Q=30 tests
# that have label prevalences in the answer key exactly equal to the Q-point
# used to compute them.
# That means each label response tuple must sum to the assumed label prevalences
# of the key
print(q_point)
[sum([joint_event_count for joint_event_count in label_responses]) for label_responses in next(iter(possible_evals))]

[10 10 10]


[np.int64(10), np.int64(10), np.int64(10)]

## The consistent set of evaluations

The consistent set of evaluations is a highly sparse subset of the possible set. On this fact rests the possible practical utility of NTQR for your work. Let us simulate a classification test by these classifiers to demonstrate how you compute consistent sets given the results of your test.

In [5]:
%pprint

Pretty printing has been turned OFF


In [6]:
# We simulate the ground truth or answer key by
# uniformly drawing from the labels
import random
answer_key = [random.choice(labels) for q in range(100)]
answer_key

['a', 'b', 'c', 'a', 'c', 'a', 'c', 'a', 'a', 'b', 'c', 'a', 'a', 'b', 'b', 'a', 'a', 'c', 'c', 'a', 'b', 'b', 'a', 'c', 'b', 'b', 'c', 'b', 'b', 'b', 'a', 'b', 'a', 'c', 'c', 'c', 'b', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'b', 'c', 'b', 'a', 'c', 'b', 'c', 'a', 'b', 'c', 'b', 'b', 'a', 'b', 'a', 'c', 'b', 'a', 'b', 'c', 'c', 'c', 'a', 'b', 'b', 'c', 'b', 'c', 'a', 'c', 'c', 'c', 'b', 'b', 'b', 'a', 'c', 'a', 'b', 'b', 'c', 'c', 'b', 'b', 'c', 'b', 'c', 'a', 'a', 'c', 'b', 'a', 'a', 'b', 'b', 'a']

In [7]:
# This answer key, like all answer keys, can be summarized
# by the count of the true labels in it. Many answer keys, not
# just this one, map to the same Q-point, as I have been calling it.
from collections import Counter
Counter(answer_key)

Counter({'b': 37, 'a': 33, 'c': 30})

In [8]:
# Now we simulate the classifiers by assuming independent decisions.
# This is merely a convenience for our simulation of a test. We could
# simulate correlations in decisions but we will not.
classifiers_accs = [{label:random.uniform(0.3,1.0) for label in labels}
                    for classifier in classifiers]
classifiers_accs

[{'a': 0.7639916138857645, 'b': 0.4094709804651383, 'c': 0.8864243612328915}, {'a': 0.6675254307810161, 'b': 0.46748322066867565, 'c': 0.36270792543325614}, {'a': 0.6845183676039116, 'b': 0.3946450912049023, 'c': 0.66792363068436}, {'a': 0.45446252379851615, 'b': 0.451263831741207, 'c': 0.7527293254863526}]

In [9]:
# Now we simulate joint decisions on the Q=100
# items to classify
test_results = [tuple(answer if random.uniform(0.0,1.0) < accs[answer]
                      else random.choice([wlabel for wlabel in labels if wlabel != answer])for accs in classifiers_accs ) 
                for answer in answer_key]
# Let's see the first 5 joint decisions
for joint_label_decisions in test_results[:5]:
    print(joint_label_decisions)

('a', 'b', 'b', 'a')
('c', 'b', 'b', 'c')
('c', 'a', 'c', 'c')
('a', 'c', 'a', 'a')
('c', 'c', 'a', 'c')


In [10]:
# NTQR erases all information about the sequence of the classifiers' responses
# just as it erases the sequence of answers in the answer key. It just wants
# the counts of the R**N ways the could have decided on the labels.
# (preview -- It turns out the semantic free nature of NTQR means it has solved
# the n-gram sequence evaluations also!)
counts = Counter(test_results)
counts

Counter({('c', 'a', 'c', 'c'): 6, ('a', 'a', 'a', 'a'): 6, ('a', 'a', 'a', 'c'): 5, ('c', 'c', 'c', 'c'): 5, ('c', 'b', 'c', 'c'): 4, ('a', 'c', 'a', 'a'): 3, ('c', 'c', 'a', 'c'): 3, ('c', 'a', 'a', 'c'): 3, ('c', 'b', 'b', 'c'): 2, ('b', 'b', 'a', 'c'): 2, ('c', 'a', 'c', 'a'): 2, ('a', 'c', 'a', 'b'): 2, ('a', 'a', 'c', 'c'): 2, ('a', 'b', 'b', 'c'): 2, ('a', 'a', 'c', 'b'): 2, ('c', 'b', 'a', 'c'): 2, ('a', 'c', 'c', 'c'): 2, ('c', 'b', 'c', 'a'): 2, ('a', 'a', 'a', 'b'): 2, ('b', 'c', 'b', 'b'): 2, ('c', 'c', 'b', 'c'): 2, ('a', 'b', 'b', 'b'): 2, ('c', 'a', 'a', 'a'): 2, ('b', 'a', 'b', 'a'): 2, ('c', 'a', 'b', 'c'): 2, ('a', 'b', 'b', 'a'): 1, ('b', 'a', 'b', 'b'): 1, ('c', 'b', 'b', 'a'): 1, ('a', 'b', 'a', 'a'): 1, ('a', 'c', 'b', 'b'): 1, ('a', 'c', 'a', 'c'): 1, ('a', 'c', 'c', 'a'): 1, ('c', 'b', 'b', 'b'): 1, ('c', 'c', 'c', 'b'): 1, ('b', 'a', 'b', 'c'): 1, ('b', 'b', 'b', 'b'): 1, ('a', 'b', 'c', 'a'): 1, ('c', 'a', 'b', 'a'): 1, ('b', 'b', 'c', 'b'): 1, ('b', 'c', 'c', 

In [11]:
# There are 81 possible ways they could decide, how
# many did we actually observe? This is one reason
# consistent sets will ALWAYS be highly sparse in the
# possible set.
len(counts)

56

The `ntqr.evaluations.ConsistentSet` has three inputs for instantiation. Labels and classifiers are the first two. The third is the observed counts for the test. Nothing else. Remember, NTQR is a logic!

In [12]:
consistent_set = ntqr.evaluations.ConsistentSet(labels, classifiers, counts)
consistent_set

ConsistentSet(('a', 'b', 'c'),('i', 'j', 'k', 'l'),Counter({('c', 'a', 'c', 'c'): 6, ('a', 'a', 'a', 'a'): 6, ('a', 'a', 'a', 'c'): 5, ('c', 'c', 'c', 'c'): 5, ('c', 'b', 'c', 'c'): 4, ('a', 'c', 'a', 'a'): 3, ('c', 'c', 'a', 'c'): 3, ('c', 'a', 'a', 'c'): 3, ('c', 'b', 'b', 'c'): 2, ('b', 'b', 'a', 'c'): 2, ('c', 'a', 'c', 'a'): 2, ('a', 'c', 'a', 'b'): 2, ('a', 'a', 'c', 'c'): 2, ('a', 'b', 'b', 'c'): 2, ('a', 'a', 'c', 'b'): 2, ('c', 'b', 'a', 'c'): 2, ('a', 'c', 'c', 'c'): 2, ('c', 'b', 'c', 'a'): 2, ('a', 'a', 'a', 'b'): 2, ('b', 'c', 'b', 'b'): 2, ('c', 'c', 'b', 'c'): 2, ('a', 'b', 'b', 'b'): 2, ('c', 'a', 'a', 'a'): 2, ('b', 'a', 'b', 'a'): 2, ('c', 'a', 'b', 'c'): 2, ('a', 'b', 'b', 'a'): 1, ('b', 'a', 'b', 'b'): 1, ('c', 'b', 'b', 'a'): 1, ('a', 'b', 'a', 'a'): 1, ('a', 'c', 'b', 'b'): 1, ('a', 'c', 'a', 'c'): 1, ('a', 'c', 'c', 'a'): 1, ('c', 'b', 'b', 'b'): 1, ('c', 'c', 'c', 'b'): 1, ('b', 'a', 'b', 'c'): 1, ('b', 'b', 'b', 'b'): 1, ('a', 'b', 'c', 'a'): 1, ('c', 'a', 'b',

## The consistent set is hugely sparse

The one message I want to leave with you is that the consistent set is highly sparse in the possible set for any Q-point. We can demonstrate that easily by looking at the Q-point extremes for a test : these are the Q-points were only one label is present in the test. At such points, there can only be one consistent evaluation. Let's confirm that.

In [13]:
%pprint

Pretty printing has been turned ON


In [14]:
# We pick a corner of the Q-simplex, only the first label is present in the answer key.
q_point = (100,0,0)
# And we get the complete consistent set using the .set_generator method
consistent_evals = [eval for eval in consistent_set.set_generator(q_point)]
consistent_evals

[(HashablePoint(<Compressed Sparse Row sparse matrix of dtype 'int64'
  	with 56 stored elements and shape (1, 81)>),
  HashablePoint(<Compressed Sparse Row sparse matrix of dtype 'int64'
  	with 0 stored elements and shape (1, 81)>),
  HashablePoint(<Compressed Sparse Row sparse matrix of dtype 'int64'
  	with 0 stored elements and shape (1, 81)>))]

So there is only one possible eval at this corner Q-point. Note how the 'b' and 'c' labels have no counts and the first label responses tuples contains counts for the event count we observed in the test results.

How many possible evaluations are at this same Q-point?

In [15]:
p_count = possible_set.set_count_at_ql(q_point)
print(p_count)
float(p_count)

30077383103880443506960119717599095978648262854822110


3.0077383103880444e+52

So the consistent set is $10^{-52}$ smaller. This is hugely sparse. Practical use of this sparsity is demonstrated in other notebooks included with the NTQR package.

## Conclusion

I hope this quick walk through will get you started with using NTQR in your work. For more examples of how to use NTQR take a look at the notebooks available when you install it.
```
~$: pip install ntqr
~$: cd <working_directory_of_choice>
~$: ntqr-docs
~$: cd ntqr_notebooks
~$: jupyter notebook
```